# 🎬 Jadi Apa — 1-Klik Colab T4 Fast Local Pipeline (`Cycles CUDA` + `OIDN GPU` + Drive Cache)

**Standar Permanen Google Colab T4 (Anti-Layar-Hitam, 0 PNG ke Drive, Baking `< 3 detik`, Cache Blender & Whisper di Drive):**
- **Mesin Render:** Pure Cycles **`CUDA` GPU ONLY** (`CPU = False`, Zero EEVEE `llvmpipe`, Zero OptiX black-frame!) + **`OPENIMAGEDENOISE` GPU (`FAST`)** + `12` adaptive samples (`0.05` / min `4`) + `1024px` texture limit + `PNG compression = 0` di `/tmp/render_frames/`.
- **Pre-Flight Guard Frame 1:** Menguji Frame 1 Scene 01 di `/tmp/` — langsung berhenti jika hitam (`brightness < 3.0`) atau 0 GPU CUDA.
- **Export Bersih:** Tanpa *top badge* ganda (hanya subtitle karaoke bawah `.ass`), salin **HANYA 1 file `final_epXXX.mp4`** ke Google Drive, lalu auto-hapus `/tmp/render_frames/`.

**Cara Pakai:** Pastikan **Runtime → Ubah jenis runtime → `GPU T4`**, cek **Sel 1**, lalu klik **Runtime → Jalankan semua**!

In [ ]:
#@title 1. Konfigurasi Episode (Cek & Klik Runtime -> Jalankan Semua) { run: "auto" }
EPISODE_ID = "ep001_kecoak_dapur" #@param {type:"string"}
RENDER_MODE = "final" #@param ["final", "preview", "storyboard"]

In [ ]:
#@title 2. 1-Klik Master Runner (Mount Drive + Cache Blender/Whisper + Pre-Flight CUDA + Export MP4 Final)
import base64
import datetime
import json
import os
import shutil
import subprocess
import time
import traceback
from collections import deque
from pathlib import Path
from IPython.display import HTML, Image as IPyImage, display
from google.colab import drive, userdata

# 0. Anti-tcmalloc crash di kontainer Docker Google Colab
os.environ["LD_PRELOAD"] = ""

REPO_URL = "https://github.com/ruangcermat/channel-micro-pov.git"
REPO_BRANCH = "main"
BLENDER_VERSION = "5.2.2"
BLENDER_TAR_NAME = f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BLENDER_URL = f"https://download.blender.org/release/Blender5.2/{BLENDER_TAR_NAME}"

# 1. Mount Google Drive & siapkan struktur folder + cache permanen Drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/channel-micro-pov")
CACHE_DIR = DRIVE_ROOT / "03_pipeline" / "cache"
WHISPER_CACHE_DIR = CACHE_DIR / "whisper"
ASSETS_DIR = DRIVE_ROOT / "02_assets"
WORK_DIR = DRIVE_ROOT / "04_episodes" / EPISODE_ID
LOCAL_FRAMES_DIR = Path(f"/tmp/render_frames/{EPISODE_ID}")
REPO_DIR = Path("/content/channel-micro-pov")
BLENDER_DIR = Path("/content/blender")
BLENDER_BIN = BLENDER_DIR / "blender"
ERROR_REPORT_PATH = WORK_DIR / "error_report.txt"

for d in (CACHE_DIR, WHISPER_CACHE_DIR, ASSETS_DIR, ASSETS_DIR / "models_3d", WORK_DIR, LOCAL_FRAMES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Arahkan cache faster-whisper / HuggingFace ke Drive agar tidak download ulang di sesi berikutnya!
os.environ["HF_HOME"] = str(WHISPER_CACHE_DIR)
os.environ["XDG_CACHE_HOME"] = str(WHISPER_CACHE_DIR)
os.environ["MICRO_POV_ASSETS_DIR"] = str(ASSETS_DIR)
os.environ["MICRO_POV_WORK_DIR"] = str(WORK_DIR)
os.environ["MICRO_POV_FRAMES_DIR"] = str(LOCAL_FRAMES_DIR)

_LAST_LOG_TAIL = deque(maxlen=150)

def save_error_report(step_name: str, exc: BaseException):
    """Simpan traceback lengkap & 150 baris log terakhir ke error_report.txt di Google Drive."""
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    tb_text = traceback.format_exc()
    tail_text = "".join(_LAST_LOG_TAIL)
    report = (
        f"=== JADI APA PIPELINE ERROR REPORT ===\n"
        f"Timestamp     : {ts}\n"
        f"Environment   : Google Colab T4 (Cycles CUDA + OIDN GPU)\n"
        f"Step          : {step_name}\n"
        f"Episode ID    : {EPISODE_ID}\n"
        f"Render Mode   : {RENDER_MODE}\n"
        f"Work Dir      : {WORK_DIR}\n"
        f"Local Frames  : {LOCAL_FRAMES_DIR}\n"
        f"Exception     : {type(exc).__name__}: {exc}\n\n"
        f"--- FULL TRACEBACK ---\n{tb_text}\n"
        f"--- LAST 150 LINES OF OUTPUT ---\n{tail_text}\n"
    )
    for dest in (ERROR_REPORT_PATH, Path("/content/error_report.txt")):
        try:
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_text(report, encoding="utf-8")
        except Exception as write_err:
            print(f"⚠️ Gagal menulis {dest}: {write_err}")
    print("\n" + "=" * 72)
    print(f"❌ ERROR PADA LANGKAH: {step_name}")
    print(f"📄 Traceback otomatis disimpan ke: {ERROR_REPORT_PATH}")
    print("=" * 72)
    print(tb_text)

def run_step(cmd, step_name: str = ""):
    """Jalankan subprocess dengan output real-time, LD_PRELOAD='', dan auto error_report.txt."""
    label = step_name or (" ".join(str(x) for x in cmd[:3]))
    _LAST_LOG_TAIL.clear()
    try:
        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"
        env["LD_PRELOAD"] = ""
        proc = subprocess.Popen(
            cmd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            _LAST_LOG_TAIL.append(line)
            print(line, end="", flush=True)
        rc = proc.wait()
        if rc != 0:
            raise subprocess.CalledProcessError(rc, cmd)
    except Exception as exc:
        save_error_report(label, exc)
        raise

def run_master_pipeline():
    t_all = time.monotonic()

    # 2. Verifikasi GPU T4 & baca Colab Secrets opsional
    print("🖥️ Memeriksa GPU Colab T4...")
    gpu_probe = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if gpu_probe.returncode != 0 or not gpu_probe.stdout.strip():
        raise RuntimeError(
            "PREFLIGHT FAILED: Black frame or no CUDA GPU — "
            "Tidak ada GPU NVIDIA aktif di Colab! Buka Runtime -> Ubah jenis runtime -> GPU T4."
        )
    print(f"✅ GPU Aktif: {gpu_probe.stdout.strip()}")

    github_token = ""
    for secret_name in ("GEMINI_API_KEY", "GITHUB_TOKEN"):
        try:
            val = userdata.get(secret_name)
            if val and val.strip():
                if secret_name == "GITHUB_TOKEN":
                    github_token = val.strip()
                else:
                    os.environ[secret_name] = val.strip()
                print(f"✅ Secret {secret_name} dimuat.")
        except Exception:
            pass

    # Deteksi Master Mas Dimas & fallback family_ready.blend di Drive
    mas_dimas_cache = ASSETS_DIR / "models_3d" / "mas_dimas_ready.blend"
    if mas_dimas_cache.exists():
        os.environ["MICRO_POV_MAS_DIMAS_BLEND"] = str(mas_dimas_cache)
        print(f"✅ Master Mas Dimas siap ({mas_dimas_cache.stat().st_size / 1e6:.1f} MB)")
    family_cache = ASSETS_DIR / "models_3d" / "family_ready.blend"
    if not family_cache.exists():
        for src in (
            Path("/content/drive/MyDrive/Skill referensi/Zack Blender Pack/Zack-Blender-Pack/zack-3d-short/ready/family_ready.blend"),
            DRIVE_ROOT / "05_skills_ref/Zack-Blender-Pack/zack-3d-short/ready/family_ready.blend",
        ):
            if src.exists():
                shutil.copy2(src, family_cache)
                break
    if family_cache.exists():
        os.environ["MICRO_POV_FAMILY_BLEND"] = str(family_cache)

    # 3. Cache Blender 5.2.2 di Google Drive -> ekstrak cepat ke /content/blender (~15 detik!)
    cached_tar = CACHE_DIR / BLENDER_TAR_NAME
    if not BLENDER_BIN.exists():
        if not cached_tar.exists() or cached_tar.stat().st_size < 100_000_000:
            tmp_tar = Path(f"/tmp/{BLENDER_TAR_NAME}")
            print(f"⬇️ Mengunduh Blender {BLENDER_VERSION} LTS (1x saja, lalu disimpan ke Cache Drive)...")
            run_step(["wget", "-q", "--show-progress", BLENDER_URL, "-O", str(tmp_tar)], "Download Blender")
            print(f"💾 Menyimpan arsip Blender ke Cache Drive: {cached_tar}")
            shutil.copy2(tmp_tar, cached_tar)
            tmp_tar.unlink(missing_ok=True)
        else:
            print(f"⚡ Cache Blender ditemukan di Drive ({cached_tar.stat().st_size / 1e6:.1f} MB): {cached_tar}")
        print(f"📦 Mengekstrak Blender ke {BLENDER_DIR} (~15 detik)...")
        BLENDER_DIR.mkdir(parents=True, exist_ok=True)
        run_step(["tar", "-xf", str(cached_tar), "-C", str(BLENDER_DIR), "--strip-components=1"], "Extract Blender")
    else:
        print(f"✅ Blender sudah siap di {BLENDER_BIN}")
    run_step([str(BLENDER_BIN), "--version"], "Blender Version")

    # 4. Clone / pull repo GitHub branch `main` + dependensi ringan
    auth_repo_url = REPO_URL.replace("https://", f"https://{github_token}@", 1) if (github_token and REPO_URL.startswith("https://")) else REPO_URL
    if (REPO_DIR / ".git").exists():
        print("🔄 Pulling update terbaru dari branch main...")
        run_step(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", auth_repo_url], "git remote")
        run_step(["git", "-C", str(REPO_DIR), "fetch", "origin"], "git fetch")
        run_step(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], "git checkout")
        run_step(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], "git pull")
    else:
        print("📥 Cloning repository GitHub...")
        run_step(["git", "clone", "-b", REPO_BRANCH, auth_repo_url, str(REPO_DIR)], "git clone")

    run_step(
        ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core fonts-liberation libxrender1 libxxf86vm1 libxfixes3 libxi6 libxkbcommon0 libsm6 libgl1 libglu1-mesa > /dev/null"],
        "apt-get install",
    )
    run_step(["pip", "install", "-q", "--upgrade", "-r", str(REPO_DIR / "03_pipeline/requirements.txt")], "pip install")

    # 5. Temukan spec episode + sinkronkan aset, cached VO, & SFX/BGM
    ep_repo_dir = REPO_DIR / "04_episodes" / EPISODE_ID
    spec_candidates = sorted(ep_repo_dir.glob("*_spec.json"))
    if not spec_candidates:
        raise FileNotFoundError(f"File *_spec.json tidak ditemukan di {ep_repo_dir}")
    spec_path = spec_candidates[0]
    print(f"📄 Episode Spec : {spec_path}")
    print(f"📁 Whisper Cache: {WHISPER_CACHE_DIR}")
    print(f"⚡ Local SSD PNG: {LOCAL_FRAMES_DIR} (0 PNG ke Drive, auto-hapus usai export!)")

    run_step(
        ["python3", str(REPO_DIR / "03_pipeline/scripts/fetch_assets.py"), "--spec", str(spec_path), "--assets-dir", str(ASSETS_DIR)],
        "fetch_assets.py",
    )
    run_step(
        [
            "python3", str(REPO_DIR / "03_pipeline/scripts/generate_tts.py"),
            "--spec", str(spec_path),
            "--work-dir", str(WORK_DIR),
            "--voice", "Puck",
            "--fps", "24",
            "--pad-sec", "0.06",
            "--vo-speed", "1.08",
            "--skip-existing",
        ],
        "generate_tts.py (cached VO)",
    )
    run_step(
        ["python3", str(REPO_DIR / "03_pipeline/scripts/prepare_audio.py"), "--spec", str(spec_path), "--assets-dir", str(ASSETS_DIR)],
        "prepare_audio.py",
    )

    # 6. Render Blender Cycles CUDA GPU ONLY + OIDN GPU FAST + Pre-Flight Guard Frame 1
    LOCAL_FRAMES_DIR.mkdir(parents=True, exist_ok=True)
    render_cmd = [
        str(BLENDER_BIN),
        "--background",
        "--python-exit-code", "1",
        "--python", str(REPO_DIR / "03_pipeline/scripts/build_and_render.py"),
        "--",
        "--cycles-device", "CUDA",
        "--spec", str(spec_path),
        "--timing", str(WORK_DIR / "timing_manifest.json"),
        "--assets-dir", str(ASSETS_DIR),
        "--work-dir", str(WORK_DIR),
        "--frames-dir", str(LOCAL_FRAMES_DIR),
        "--mode", RENDER_MODE,
        "--engine", "CYCLES_CUDA",
    ]
    print("🚀 Menjalankan Blender 5.2.2 (Pure Cycles CUDA + OIDN GPU FAST + Pre-Flight Guard):")
    print("  ", " ".join(render_cmd))
    run_step(render_cmd, f"Blender Cycles CUDA Render ({RENDER_MODE})")

    if RENDER_MODE == "storyboard":
        sheets = sorted(WORK_DIR.glob("storyboard_sheet_*.png")) or ([WORK_DIR / "storyboard_sheet.png"] if (WORK_DIR / "storyboard_sheet.png").exists() else [])
        for sp in sheets:
            print(f"🎬 Storyboard Sheet: {sp}")
            display(IPyImage(filename=str(sp)))
        return

    # 7. Compose Final MP4 (FFmpeg Post-Look + Clean Bottom Karaoke .ass + 4-Layer Audio -14 LUFS + Auto-Cleanup /tmp/)
    run_step(
        [
            "python3", str(REPO_DIR / "03_pipeline/scripts/compose_final.py"),
            "--spec", str(spec_path),
            "--timing", str(WORK_DIR / "timing_manifest.json"),
            "--assets-dir", str(ASSETS_DIR),
            "--work-dir", str(WORK_DIR),
            "--frames-dir", str(LOCAL_FRAMES_DIR),
            "--mode", RENDER_MODE,
            "--cleanup-frames",
        ],
        "compose_final.py (FFmpeg Export + Auto-Cleanup /tmp/)",
    )

    # 8. Verifikasi output akhir di Google Drive & tampilkan preview player
    manifest = json.loads((WORK_DIR / "timing_manifest.json").read_text(encoding="utf-8"))
    ep_code = manifest["episode_id"].lower()
    out_name = f"final_{ep_code}.mp4" if RENDER_MODE == "final" else f"preview_{ep_code}.mp4"
    video_path = WORK_DIR / out_name
    assert video_path.exists(), f"Output MP4 tidak ditemukan: {video_path}"

    size_mb = video_path.stat().st_size / (1024 * 1024)
    total_min = (time.monotonic() - t_all) / 60.0
    print("\n" + "=" * 72)
    print(f"🎉 SELESAI DALAM {total_min:.1f} MENIT!")
    print(f"📂 HANYA 1 file MP4 tersimpan permanen di Drive: {video_path} ({size_mb:.2f} MB)")
    if not LOCAL_FRAMES_DIR.exists():
        print(f"🧹 Disk lokal bersih: {LOCAL_FRAMES_DIR} sudah otomatis dihapus!")
    print("=" * 72)

    if size_mb <= 35.0:
        b64 = base64.b64encode(video_path.read_bytes()).decode("ascii")
        display(HTML(
            '<div style="display:flex;justify-content:center;padding:12px;background:#111;border-radius:12px;">'
            '<video width="360" height="640" controls playsinline style="border-radius:10px;">'
            f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
            "</video></div>"
        ))

try:
    run_master_pipeline()
except Exception as _exc:
    save_error_report("Master Pipeline", _exc)
    raise